## 1. Background

We split participants into two equally sized teams. We want complementary skills within each team and balanced total skill scores across the two teams.

In a hackathon, teams may need programming, design, and communication skills to build and present a working prototype. Manual assignment depends on the organizer's judgment, random assignment can leave uneven skill coverage, and optimizing a single score can concentrate one skill in one team. We use an explicit objective to make these tradeoffs visible and reproducible. Our simplified model treats differences between skill profiles as a proxy for complementarity.

We use **1,000 synthetic participants**. Programming, Design, and Communication scores are integers from 0 to 3, with master seed `20261014`. We select a fixed group of 20 participants and a fixed subset of 6 from that group for the two experiments. These scores illustrate the mathematical model; they are not assessments of real people.

We generate the data, formulate the model, run actual Qiskit QAOA, validate the results classically, and export the same results to a static website.

In [1]:
import os
# We fix the thread environment before importing NumPy / Qiskit.
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "RAYON_NUM_THREADS", "QISKIT_NUM_PROCS"):
    os.environ[name] = "1"

import sys
from pathlib import Path
from html import escape
from IPython.display import HTML, display

cwd = Path.cwd().resolve()
candidates = (cwd, cwd / "jupyter", cwd.parent / "jupyter")
HERE = next((p for p in candidates if (p / "model.py").is_file()
             and (p / "solvers.py").is_file()), None)
if HERE is None:
    raise FileNotFoundError("Open the notebook from the project root or the jupyter directory.")
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import json
import math
import numpy as np
from model import (generate_data, load_cases, pool_summary, encoding,
                   verify_small, evaluate, penalty, qubo, to_ising)
from solvers import run_quantum, classical_results, export_results

DATA_DIR = HERE / "data"
EXPORT = HERE.parent / "frontend" / "data" / "results.json"

def fmt(value):
    if value is None:
        return "—"
    if isinstance(value, (float, np.floating)):
        return f"{value:.6g}"
    return str(value)

def table(headers, rows):
    def row(values, tag):
        return "<tr>" + "".join(f"<{tag}>{escape(fmt(v))}</{tag}>" for v in values) + "</tr>"
    display(HTML("<div style='overflow-x:auto'><table style='border-collapse:collapse;"
                 "font-size:14px;text-align:left' border='1' cellpadding='7'>"
                 + row(headers, "th") + "".join(row(r, "td") for r in rows)
                 + "</table></div>"))

print("Working directory:", HERE)


Working directory: /Users/jenny/Documents/Codex/2026-10-11/u/outputs/quantum-team-builder-simple/jupyter


In [2]:
# We explicitly generate reproducible data files; reruns write the same fixed data.
generate_data(DATA_DIR)
cases = load_cases(DATA_DIR)
pool = pool_summary(DATA_DIR)
assert [case["n"] for case in cases] == [6, 20]
assert set(cases[0]["ids"]).issubset(cases[1]["ids"])
print(f"Synthetic participants: {pool['rows']}; master seed: {pool['master_seed']}; data SHA-256: {pool['sha256']}")
table(["Participant", *pool["skills"]],
      [[r["id"], *[r[s] for s in pool["skills"]]] for r in pool["head"]])


Synthetic participants: 1000; master seed: 20261014; data SHA-256: 14ddc37ca98ef3d82e00f83348ff09574598094362add99b44c0920d637e7ffb


Participant,Programming,Design,Communication
P0001,0,2,0
P0002,1,0,1
P0003,3,3,2
P0004,0,2,1
P0005,2,1,3
P0006,1,0,2


## 2. Objective and Constraints

We use $x_i=1$ to assign participant $i$ to Team 1 and $x_i=0$ to assign them to Team 2. We require $\sum_i x_i=n/2$. Let $a_{is}$ be the skill score, $S_s=\sum_i a_{is}$, and $c_{ij}=\frac{1}{3}\sum_s|a_{is}-a_{js}|$ be the difference between two skill profiles.

We reward complementary profiles within each team and penalize deviations from half of the total score for each skill:

$$C(x)=\sum_{i<j}c_{ij}[1-x_i-x_j+2x_ix_j],\qquad
B(x)=2\sum_s\left(\sum_i a_{is}x_i-S_s/2\right)^2.$$

We minimize
$$E(x)=-C(x)+B(x)+P\left(\sum_i x_i-n/2\right)^2.$$

We fix the complementarity and skill-balance weights to 1. We automatically set a safe team-size penalty:
$$P=C_{\max}+B_{\max}+1,\quad C_{\max}=\sum_{i<j}c_{ij},\quad B_{\max}=\tfrac12\sum_sS_s^2.$$
Every infeasible assignment incurs a penalty of at least $P$, making its energy strictly higher than the upper bound for any feasible assignment. The encoded global optimum therefore satisfies the team-size constraint. A QAOA run with a finite budget can still return an infeasible assignment; we preserve and label that outcome.

In [3]:
for case in cases:
    print(f"{case['n']}-person case: {case['n'] // 2} participants per team; P = {penalty(case):.6g}")
    table(["Participant", *case["skills"]],
          [[pid, *scores] for pid, scores in zip(case["ids"], case["scores"])])


6-person case: 3 participants per team; P = 140.667


Participant,Programming,Design,Communication
P0484,0,0,3
P0766,2,3,3
P0138,2,0,1
P0605,2,1,0
P0848,3,2,2
P0623,2,0,0


20-person case: 10 participants per team; P = 1815.17


Participant,Programming,Design,Communication
P0757,0,3,2
P0754,1,0,3
P0027,1,2,0
P0476,3,0,3
P0766,2,3,3
P0605,2,1,0
P0151,2,2,1
P0484,0,0,3
P0138,2,0,1
P0292,2,1,2


## 3. QUBO

We expand the same energy into an unconstrained binary quadratic model:
$$E(x)=q_0+\sum_i h_ix_i+\sum_{i<j}J_{ij}x_ix_j.$$

With $m=n/2$, we obtain
$$q_0=-C_{\max}+\tfrac12\sum_sS_s^2+Pm^2,$$
$$h_i=\sum_{j\ne i}c_{ij}+2\sum_s(a_{is}^2-a_{is}S_s)+P(1-2m),$$
$$J_{ij}=-2c_{ij}+4\sum_sa_{is}a_{js}+2P.$$

We retain the constant term so that direct evaluation, QUBO, and Ising energies can be compared consistently. The colored tables show the upper-triangular quadratic coefficients. For the 20-person case, we preview the coefficients between the first 6 participants; we save all coefficients in the exported JSON.

In [4]:
encodings = {case["id"]: encoding(case) for case in cases}
for case in cases:
    q = encodings[case["id"]]["qubo"]
    print(f"{case['n']} participants: QUBO constant q0 = {q['constant']:.6g}")
    table(["Variable", "Participant", "Linear coefficient h_i"],
          [[f"x_{i}", pid, q["linear"][i]] for i, pid in enumerate(case["ids"])])
    J = np.asarray(q["quadratic"])
    width = min(6, case["n"])
    scale = max(float(np.max(np.abs(J))), 1.0)
    header = "<tr><th>Jᵢⱼ</th>" + "".join(f"<th>x_{j}</th>" for j in range(width)) + "</tr>"
    rows = []
    for i in range(width):
        row = f"<tr><th>x_{i}</th>"
        for j in range(width):
            value = J[i, j]
            color = f"rgba(53,110,184,{0.08 + 0.55 * abs(value) / scale:.3f})"
            row += (f"<td style='background:{color}'>{value:.4g}</td>" if j > i else "<td>—</td>")
        rows.append(row + "</tr>")
    display(HTML("<table border='1' cellpadding='7' style='border-collapse:collapse'>"
                 + header + "".join(rows) + "</table>"))


6 participants: QUBO constant q0 = 1364.33


Variable,Participant,Linear coefficient h_i
x_0,P0484,-730.667
x_1,P0766,-785.333
x_2,P0138,-750
x_3,P0605,-743.333
x_4,P0848,-788
x_5,P0623,-733.333


Jᵢⱼ,x_0,x_1,x_2,x_3,x_4,x_5
x_0,—,314,290.7,277.3,301.3,278
x_1,—,—,306,306,351.3,293.3
x_2,—,—,—,296,310.7,296.7
x_3,—,—,—,—,310.7,296.7
x_4,—,—,—,—,—,302
x_5,—,—,—,—,—,—


20 participants: QUBO constant q0 = 182868


Variable,Participant,Linear coefficient h_i
x_0,P0757,-34742.5
x_1,P0754,-34721.8
x_2,P0027,-34633.8
x_3,P0476,-34839.8
x_4,P0766,-34933.8
x_5,P0605,-34646.5
x_6,P0151,-34769.2
x_7,P0484,-34651.2
x_8,P0138,-34661.8
x_9,P0292,-34784.5


Jᵢⱼ,x_0,x_1,x_2,x_3,x_4,x_5
x_0,—,3651,3652,3650,3688,3638
x_1,—,—,3631,3677,3672,3635
x_2,—,—,—,3638,3659,3645
x_3,—,—,—,—,3688,3651
x_4,—,—,—,—,—,3655
x_5,—,—,—,—,—,—


## 4. Ising

We map the variables to qubits using $x_i=(1-Z_i)/2$. Exchanging the team labels leaves the objective unchanged, so the one-body $Z_i$ terms cancel:
$$H_C=\text{offset}\,I+\frac14\sum_{i<j}J_{ij}Z_iZ_j,\qquad
\text{offset}=q_0+\frac12\sum_i h_i+\frac14\sum_{i<j}J_{ij}.$$

We use the Pauli operator without its constant term for circuit evolution and restore the offset when comparing energies. Qiskit prints bitstrings in the order $x_{n-1}\ldots x_0$, while our participant list follows $x_0,\ldots,x_{n-1}$. We decode the strings accordingly.

We display the generated Pauli terms and the parameterized 6-qubit QAOA circuit with $p=1$. This circuit illustrates the structure; we perform actual training and measurement in the next section.

In [5]:
for case in cases:
    op, offset = to_ising(case)
    terms = op.to_list()
    print(f"{case['n']} qubits; offset = {offset:.6g}; Pauli terms = {len(terms)}")
    table(["Pauli string", "Coefficient"],
          [[label, float(np.real(coef))] for label, coef in terms[:15]])
    if len(terms) > 15:
        print("We show the first 15 terms here and save all Pauli terms in the exported JSON.")

from qiskit.circuit.library import QAOAAnsatz
small_op, _ = to_ising(cases[0])
illustration = QAOAAnsatz(cost_operator=small_op, reps=1)
print(illustration.decompose().draw(output="text", fold=110))


6 qubits; offset = 231.667; Pauli terms = 15


Pauli string,Coefficient
IIIIZZ,78.5
IIIZIZ,72.6667
IIZIIZ,69.3333
IZIIIZ,75.3333
ZIIIIZ,69.5
IIIZZI,76.5
IIZIZI,76.5
IZIIZI,87.8333
ZIIIZI,73.3333
IIZZII,74


20 qubits; offset = 9072.5; Pauli terms = 190


Pauli string,Coefficient
IIIIIIIIIIIIIIIIIIZZ,912.75
IIIIIIIIIIIIIIIIIZIZ,912.917
IIIIIIIIIIIIIIIIZIIZ,912.417
IIIIIIIIIIIIIIIZIIIZ,922.083
IIIIIIIIIIIIIIZIIIIZ,909.583
IIIIIIIIIIIIIZIIIIIZ,914.917
IIIIIIIIIIIIZIIIIIIZ,912.917
IIIIIIIIIIIZIIIIIIIZ,908.583
IIIIIIIIIIZIIIIIIIIZ,913.917
IIIIIIIIIZIIIIIIIIIZ,906.417


We show the first 15 terms here and save all Pauli terms in the exported JSON.
     ┌───┐»
q_0: ┤ H ├»
     ├───┤»
q_1: ┤ H ├»
     ├───┤»
q_2: ┤ H ├»
     ├───┤»
q_3: ┤ H ├»
     ├───┤»
q_4: ┤ H ├»
     ├───┤»
q_5: ┤ H ├»
     └───┘»
«     ┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐»
«q_0: ┤0                                                                                                                                                       ├»
«     │                                                                                                                                                        │»
«q_1: ┤1                                                                                                                                                       ├»
«     │                                                                                                              

## 5. QAOA

We run actual `QAOA` + `MinimumEigenOptimizer` sequentially, using an ideal `StatevectorSampler`, COBYLA, and fixed random seeds. We set $p=1$ and use 2,048 shots per objective evaluation. We allow at most 200 objective evaluations for 6 participants and 60 for 20 participants, with a 300-second limit for each case. Recorded wall time also includes process startup, final measurement, and related overhead.

We preserve the solver's returned assignment unchanged. We then measure the trained circuit for another 2,048 shots using an independent seed to estimate feasible and optimal assignment frequencies. We do not replace the solver's returned assignment with a result from these independent measurements. We calculate optimal-assignment frequency only after the classical exact solution is available; the exact answer is not used during quantum training.

We rerun both cases without loading previous results. We retain timeouts, errors, and infeasible assignments without replacing them with classical answers. The displayed frequencies are empirical finite-shot estimates, not exact probabilities.

In [6]:
quantum_results = []
for case in cases:
    print(f"Starting {case['n']}-person QAOA; seed {case['qaoa_seed']}; limit 300 seconds...", flush=True)
    result = run_quantum(case, wall_seconds_limit=300)
    quantum_results.append(result)
    print(f"Status: {result['status']}; elapsed: {result['runtime_seconds']:.3f} seconds; "
          f"objective evaluations: {result['budgets']['objective_evaluations']}; "
          f"stop reason: {result['qaoa'].get('stop_reason')}")
    if result.get("error"):
        print("Original error:", result["error"])
    returned = result["returned"]
    if returned is None:
        print("The solver did not return an assignment for this run.")
    else:
        table(["Feasible", "Complementarity C", "Skill balance B", "Size penalty", "Total energy E"],
              [[returned["feasible"], returned["C"], returned["B"],
                returned["size_penalty"], returned["E"]]])
        table(["Team", "Participant", *case["skills"]],
              [[team, ", ".join(returned[team]),
                *[returned["skill_sums"][team][s] for s in case["skills"]]]
               for team in ("team_1", "team_2")])
    counts = result["qaoa"]["counts"]
    total = sum(counts.values())
    if total:
        print(f"Independent final measurement: {total} shots; feasible-assignment frequency: "
              f"{result['qaoa']['feasible_sample_mass']:.2%}. We show at most 10 entries below; frequencies use all shots as the denominator.")
        table(["Qiskit printed bitstring", "Count", "Empirical frequency"],
              [[bits, count, f"{count / total:.2%}"]
               for bits, count in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:10]])


Starting 6-person QAOA; seed 3036182775; limit 300 seconds...


Status: completed; elapsed: 0.923 seconds; objective evaluations: 26; stop reason: optimizer_terminated


Feasible,Complementarity C,Skill balance B,Size penalty,Total energy E
True,9.33333,1,0,-8.33333


Team,Participant,Programming,Design,Communication
team_1,"P0484, P0605, P0848",5,3,5
team_2,"P0766, P0138, P0623",6,3,4


Independent final measurement: 2048 shots; feasible-assignment frequency: 41.55%. We show at most 10 entries below; frequencies use all shots as the denominator.


...

Status: completed; elapsed: 268.185 seconds; objective evaluations: 24; stop reason: optimizer_terminated


Feasible,Complementarity C,Skill balance B,Size penalty,Total energy E
True,114.333,0.5,0,-113.833


Team,Participant,Programming,Design,Communication
team_1,"P0754, P0027, P0138, P0292, P0623, P0147, P0971, P0740, P0848, P0177",17,14,17
team_2,"P0757, P0476, P0766, P0605, P0151, P0484, P0132, P0325, P0347, P0264",17,14,18


Independent final measurement: 2048 shots; feasible-assignment frequency: 17.33%. We show at most 10 entries below; frequencies use all shots as the denominator.


Qiskit printed bitstring,Count,Empirical frequency
01001000001011101010,2,0.10%
01011100000001011100,2,0.10%
00000000001010011111,1,0.05%
00000000010001011000,1,0.05%
00000000010001111101,1,0.05%
00000000011000010101,1,0.05%
00000000011001100110,1,0.05%
00000000100011010101,1,0.05%
00000000100101110011,1,0.05%
00000000110011101011,1,0.05%


## 6. Classical Verification

We enumerate all 64 binary states of the 6-person case and independently compare direct scoring, QUBO, the Qiskit objective, and Ising energies. We also check bit ordering and the feasibility of all ground states. We then enumerate every feasible assignment as an exact reference: $\binom63=20$ labeled assignments for 6 participants and $\binom{20}{10}=184,756$ for 20 participants. Swapping team labels represents the same unlabeled partition, but we consistently include both labelings in all counts.

We also search 100 uniformly random feasible assignments using a fixed seed as a simple baseline. We report $\mathrm{gap}=E_{\mathrm{returned}}-E^*$ only for feasible returned assignments; missing values are not treated as zero. The methods have different computational budgets, so we do not claim quantum speedup or quantum advantage.

We export both cases, actual solver statuses, objective components, full coefficients, training histories, and final sample counts. The static website reads this JSON and does not run quantum computation in the browser.

In [7]:
verification = verify_small(cases[0])
assert verification["passed"] and verification["ground_states_feasible"]
assert verification["bit_order_roundtrip"]
table(["Verification check", "Result"],
      [["Enumerated states", verification["n_states"]],
       ["Feasible states", verification["n_feasible"]],
       ["Maximum energy error", verification["max_error"]],
       ["All ground states feasible", verification["ground_states_feasible"]],
       ["Bit-order round trip passed", verification["bit_order_roundtrip"]]])

classic_results = []
for case in cases:
    print(f"Classical verification for {case['n']} participants: enumerating {math.comb(case['n'], case['n'] // 2):,} feasible labeled assignments.")
    classic_results.append(classical_results(case))

payload = export_results(cases, quantum_results, classic_results, DATA_DIR, EXPORT)
rows = []
for case in payload["cases"]:
    for result in case["results"]:
        returned = result["returned"] or {}
        rows.append([case["n"], result["solver"], result["status"],
                     returned.get("feasible"), returned.get("C"), returned.get("B"),
                     returned.get("E"), result["gap"], result["runtime_seconds"]])
table(["Participants", "Method", "Status", "Feasible", "C", "B", "E", "gap", "Time / seconds"], rows)


Verification check,Result
Enumerated states,64
Feasible states,20
Maximum energy error,1.02318e-12
All ground states feasible,True
Bit-order round trip passed,True


Classical verification for 6 participants: enumerating 20 feasible labeled assignments.
Classical verification for 20 participants: enumerating 184,756 feasible labeled assignments.


Participants,Method,Status,Feasible,C,B,E,gap,Time / seconds
6,exact,optimal,True,9.33333,1,-8.33333,0,0.000134625
6,random,completed,True,9.33333,1,-8.33333,0,0.000448916
6,qaoa,completed,True,9.33333,1,-8.33333,0,0.922851
20,exact,optimal,True,115,0.5,-114.5,0,0.0621164
20,random,completed,True,114.333,2.5,-111.833,2.66667,0.000378458
20,qaoa,completed,True,114.333,0.5,-113.833,0.666667,268.185


In [8]:
quantum_rows = []
for case in payload["cases"]:
    result = next(r for r in case["results"] if r["solver"] == "qaoa")
    q = result["qaoa"]
    resources = q["resources"]
    quantum_rows.append([case["n"], result["status"], sum(q["counts"].values()),
                         q["feasible_sample_mass"], q["optimal_sample_mass"],
                         resources["depth"], resources["cx_count"]])
table(["Participants", "Status", "Independent final shots", "Feasible frequency", "Optimal frequency", "Decomposed depth", "CX count"], quantum_rows)
print("Results written to:", EXPORT)
print("Versions and computing environment:")
print(json.dumps(payload["environment"], ensure_ascii=False, indent=2))
print("We use synthetic data and ideal simulation; these cases validate the model and do not demonstrate quantum advantage.")


Participants,Status,Independent final shots,Feasible frequency,Optimal frequency,Decomposed depth,CX count
6,completed,2048,0.415527,0.0253906,30,30
20,completed,2048,0.17334,0,114,380


Results written to: /Users/jenny/Documents/Codex/2026-10-11/u/outputs/quantum-team-builder-simple/frontend/data/results.json
Versions and computing environment:
{
  "python": "3.12.14",
  "platform": "macOS-26.3.1-arm64-arm-64bit",
  "machine": "arm64",
  "processor": "arm",
  "threads": 1,
  "versions": {
    "qiskit": "2.1.2",
    "qiskit-optimization": "0.7.0",
    "numpy": "2.5.4",
    "scipy": "1.18.1"
  }
}
We use synthetic data and ideal simulation; these cases validate the model and do not demonstrate quantum advantage.
